# Staged history matching with pestpp-ies, through the API

This notebook does something the `pestpp-ies` executable cannot: it runs a few iterations
against one set of observations, **changes which observation weights**, and
carries on with the same ensemble for more iterations.

That is a normal workflow in groundwater modeling that is more bespoke in practice. 
You rarely want every observation fighting for influence from
iteration one - heads and fluxes are on different scales, they constrain different things, and
throwing them all in at once tends to let the loudest group dominate. And we guess at which 
observations should get which weights before adjusting parameters and learning more about which 
obs can and cannot be fit...

The reason this needs the API is not that pest++ cannot do it - it is that the intervention
happens between iterations, and the built-in loop has no gap to put this action in and, more importantly
this type of intervention is very problem specific - it is impossible to design a generic scheme
to cover all the possible usecases (well, not impossible, but it would be very ugly!). 

**What this demonstrates**

1. Run iterations with only the head observations weighted.
2. Switch the streamflow (`gage`) observations **on**, through an explicit call.
3. Optionally **reinflate** the parameter ensemble at the same moment, so it has the spread to
   respond to the new data.
4. Carry on iterating, and watch phi decompose by group.

The model is the MODFLOW 6 Freyberg synthetic - small, fast, and the standard teaching case.

## Setup

Two knobs at the top. `REINFLATE` is the interesting one and is discussed properly further
down - leave it `True` the first time, then re-run the notebook with it `False` and compare.

In [ ]:
import os
import shutil
import sys

import numpy as np
import pandas as pd
import pyemu

sys.path.insert(0, os.path.join("..", "python"))
from pestpp import Ies

N_REALS   = 10        # small so the notebook runs in well under a minute

BENCH = os.path.join("..", "benchmarks")
# the model binary has to be findable by the forward run
os.environ["PATH"] += os.pathsep + os.path.abspath(
    os.path.join(BENCH, "test_bin",
                 "win" if os.name == "nt" else
                 ("mac" if sys.platform == "darwin" else "linux")))

workdir = "staged_master"
if os.path.exists(workdir):
    shutil.rmtree(workdir)
shutil.copytree(os.path.join(BENCH, "mf6_freyberg", "template"), workdir)
print("working directory:", workdir)

## Split the observations into two stages

The Freyberg case has 36 weighted observations: 24 groundwater levels at two sites
(`trgw_*`) and 12 streamflow values at the gage. Stage one uses the heads; stage two adds the
gage.

Setting the gage weights to zero is what makes them *inactive* - and inactive means more than
"contributes nothing to phi". An observation with zero weight is left out of the active set
entirely: it gets no column in the weights ensemble, and - the part that is easy to forget -
**no noise realizations are drawn for it**, because there was nothing to draw them from.
Switching it back on later has to put all of that back, which is why it needs a real call
rather than just an assignment.

In [ ]:
pst_file = "freyberg6_run_ies.pst"
pst = pyemu.Pst(os.path.join(workdir, pst_file))

groups = pst.observation_data.loc[pst.nnz_obs_names, "obgnme"]
head_obs = [n for n in pst.nnz_obs_names if groups[n].startswith("trgw")]
flux_obs = [n for n in pst.nnz_obs_names if groups[n] == "gage"]
print("stage 1 - heads     :", len(head_obs))
print("stage 2 - streamflow:", len(flux_obs))

# remember what the flux weights were, so stage two can restore them rather than invent them
flux_weights = pst.observation_data.loc[flux_obs, "weight"].astype(float).to_dict()

pst.observation_data.loc[flux_obs, "weight"] = 0.0        # stage one: heads only
pst.control_data.noptmax = 1
pst.pestpp_options["ies_num_reals"] = N_REALS * 2
pst.pestpp_options["random_seed"] = 11
pst.pestpp_options["ies_no_noise"] = False

# This case ships with localization switched on, and localization is resolved against the
# ACTIVE observation set. A localizer built while the gage observations were switched off has
# no rows for them, so it has to be told to tolerate that - or, as here, left out of a demo
# that is about something else. See the closing notes.
pst.pestpp_options.pop("ies_localizer", None)
pst.pestpp_options.pop("ies_autoadaloc", None)

pst.write(os.path.join(workdir, pst_file), version=2)
print("wrote", pst_file)

## Stage one: heads only

Nothing unusual here - open a session, initialize, iterate. The only thing worth noticing is
what `initialize()` decided the active set was.

In [ ]:
history = []      # (stage, iteration, phi_mean) as we go

ies = Ies.from_pst(pst_file, workdir=workdir,ies_num_reals=50,ies_reinflate_num_reals=10)
ies.initialize()

#make these first few iters cheap
ies.set_option("ies_lambda_mults", "1.0")
ies.set_option("lambda_scale_fac", "1.0")

active = list(ies.weights_df(lower=True).columns)
print("realizations      :", ies.n_reals)
print("active observations:", len(active), "(all heads:",
      all(a.startswith("trgw") for a in active), ")")
print("initial phi        : {0:.4g}".format(ies.phi))
initial_phi = ies.phi
history.append(("stage 1", 0, ies.phi))

In [ ]:
for _ in range(10):
    step = ies.solve()
    history.append(("stage 1", step.iter, step.phi_mean))
    print("iteration {0}: phi_mean {1:.4g}".format(step.iter, step.phi_mean))
    if step.phi_mean < initial_phi / 10:
        print("mean phi < 1/10 * initial_phi, breaking") 
        break

## The switch

This is the part the executable has no room for.

`set_obs_weights` here is doing more than assigning numbers. Because these observations were
at zero weight, they are being **activated**: they join the active set, gain a column in the
weights ensemble, and get noise realizations generated for them using the same draw the
initial ensemble used - same ordering, same grouping, same generator. Without that last part
they would be compared against nothing.

Note what is deliberately *not* coupled: changing the weight of an observation that was
already active does **not** touch its noise. Weights and noise are independent, and redrawing
noise on every weight change would make this iteration's phi incomparable with the last one's.
Only the off-to-on transition is structural, because only then is there no noise to preserve.

In [ ]:
flux_weights

In [ ]:
ies.noise_df()

In [ ]:
ies.set_obs_weights(flux_weights)          # <-- the whole switch

Now check that the noise ensemble has been expanded:

In [ ]:
ies.noise_df()

In [ ]:
ies.weights_df()

## Reinflation

In [ ]:

spread_before = ies.par_df(lower=True).std().mean()
ies.reinflate(factor=1.0,num_reals=20)
spread_after = ies.par_df(lower=True).std().mean()
print("mean parameter std: {0:.4g} -> {1:.4g}  ({2:.1f}x)".format(
    spread_before, spread_after, spread_after / spread_before))


In [ ]:
ies.obs_df()

In [ ]:
ies.par_df()

## Stage two

Recompute phi first, so the jump caused by the new observations is visible on its own rather
than mixed in with the effect of an iteration.

In [ ]:
ies.update_phi()
print("phi with the new observations included: {0:.4g}".format(ies.phi))
history.append(("switch", history[-1][1], ies.phi))
#increase the solver power here too
ies.set_option("ies_lambda_mults",(0.1,1,10))
ies.set_option("ies_bad_phi_sigma",1.4)

for _ in range(5):
    step = ies.solve()
    history.append(("stage 2", step.iter, step.phi_mean))
    print("iteration {0}: phi_mean {1:.4g}".format(step.iter, step.phi_mean))

## What happened, by group

`phi_group_df` decomposes phi the way the tools' own "observation group phi summary" does, but
as a frame. This is the payoff: the gage contribution appears from nothing at the switch, and
both groups are then fitted together.

In [ ]:
grp = ies.phi_group_df(lower=True)
summary = grp.mean().sort_values(ascending=False)
print("mean phi contribution by group, final ensemble:")
print(summary.to_string())

print()
print("phi history:")
print(pd.DataFrame(history, columns=["stage", "iteration", "phi"]).to_string(index=False))

In [ ]:
ies.finalize()
ies.close()
print("done - outputs are in", workdir)

## Notes and caveats

**Localization is switched off in this notebook, on purpose.** A localizer is resolved against
the active observation set when it is built, so one built while the gage observations were
inactive has no rows for them. The library re-reads the localizer when observations are
activated, which handles the file-based case; `ies_autoadaloc` rebuilds its own matrix during
the solve and does not yet cope with the active set growing underneath it. If you need
localization *and* staging today, use a file-based localizer with
`ies_localizer_forgive_missing` set, and check the `.rec` file.

**Phi is not comparable across the switch.** Stage one's phi is measured over 24 observations
and stage two's over 36, so the jump at the switch is arithmetic, not a failure. Compare
within a stage, or compare per-group contributions.

**Reinflation discards information.** It deliberately puts back variance the assimilation had
removed, which is the point - but if the ensemble had genuinely converged on the right answer,
you are throwing some of that away. It is a tool for when you believe the narrowing was
premature, which is exactly what bringing in unseen data implies.

**On this case, over two iterations, it is close to a wash** - and that is worth saying rather
than letting the notebook imply otherwise. Running it both ways gives roughly:

| | at the switch | iteration 3 | iteration 4 |
|---|---|---|---|
| `REINFLATE = True` | 1234 | 210 | 165 |
| `REINFLATE = False` | 238 | 214 | 166 |

The reinflated ensemble starts much worse - it has just been given back its prior spread, so
of course it fits less well - and then improves faster, ending in the same place. Two
iterations is not enough to separate them. What the reinflated run *has* bought is spread: it
reaches iteration 4 with an ensemble that can still move, where the un-reinflated one is
running out of room. Whether that pays for itself depends on how many iterations follow and
how much the new data really disagrees with the old, which is a judgement about your problem
rather than something a default can make for you.